# Customer Intelligence Platform — Notebook 4: MLflow Tracking & Model Registry
**Portfolio Project | Notebook 5 of 5**

---

## Learning Objectives
1. Log every churn-model candidate from Notebook 3 — parameters, metrics, and the fitted pipeline artefact — to MLflow Tracking, not just the winner
2. Encode this project's quality gate (a minimum recall and ROC-AUC the model must clear before it's deployable) as an explicit, queryable MLflow tag, not a decision made only in your head
3. Register the best gate-passing pipeline to the MLflow Model Registry and promote it through the `Staging → Production` stage lifecycle
4. Understand exactly what a FastAPI service or CLI batch job (this project's serving layer) would load from the registry, and why that decouples "deploying a new model" from "deploying new code"

> **Senior engineer framing:** Notebook 3 already produced a solid, leakage-free pipeline — but "it ran in my notebook and got 0.87 AUC" is not the same thing as a governed, reproducible, promotable artefact. This notebook is what turns a good notebook result into something a serving layer can point at with confidence, and something a teammate (or you, in six months) can audit: which exact hyperparameters, on which exact data split, cleared the bar to go live.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import recall_score, precision_score, roc_auc_score, f1_score

try:
    import mlflow
    import mlflow.sklearn
    from mlflow.tracking import MlflowClient
    MLFLOW_AVAILABLE = True
except ImportError:
    MLFLOW_AVAILABLE = False
    print("mlflow not installed — run `pip install mlflow` to enable the tracking sections.")

RNG_SEED = 42
DATA_PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")

EXPERIMENT_NAME = "customer-intelligence-platform-churn"

if MLFLOW_AVAILABLE:
    mlflow.set_experiment(EXPERIMENT_NAME)

---
## Part 1: Rebuild the Leak-Free Pipeline Components from Notebook 3

Each notebook in this project is self-contained and reproducible on its own — so the custom transformer and `ColumnTransformer` are redefined here rather than imported from another notebook. In a production codebase, this is exactly the logic that would be extracted once into `src/pipeline/` and imported everywhere, instead of being copy-pasted per notebook — see the Senior Engineer Notes at the end.

In [ ]:
class RecencyRiskFlagger(BaseEstimator, TransformerMixin):
    def __init__(self, percentile: float = 75):
        self.percentile = percentile

    def fit(self, X: pd.DataFrame, y=None):
        self.threshold_ = np.percentile(X["recency_days"], self.percentile)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X["long_inactive_flag"] = (X["recency_days"] > self.threshold_).astype(int)
        return X


df = pd.read_csv(DATA_PROCESSED_DIR / "customers_with_segments.csv")
df["segment"] = df["segment"].astype(str)

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["churned"], random_state=RNG_SEED)
target_col = "churned"
drop_cols = ["customer_id", target_col]
X_train, y_train = train_df.drop(columns=drop_cols), train_df[target_col]
X_test, y_test = test_df.drop(columns=drop_cols), test_df[target_col]

numeric_cols = [
    "tenure_months", "monthly_spend", "total_spend_lifetime", "recency_days",
    "frequency_12m", "support_tickets_12m", "discount_usage_rate",
    "avg_session_minutes", "num_products", "satisfaction_score", "long_inactive_flag",
]
categorical_cols = ["contract_type", "region", "acquisition_channel", "payment_method", "segment"]

numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
categorical_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("num", numeric_pipe, numeric_cols), ("cat", categorical_pipe, categorical_cols)])

---
## Part 2: The Quality Gate — Defined Once, Applied Consistently

Every run gets tagged `passes_quality_gate` based on the **same** objective threshold, so the registry decision is never made by eyeballing a metrics table.

### TODO 1 — Define the gate thresholds and the function that applies them

**HINT:** for a churn use case, recall on the churned class matters more than raw accuracy (missing an about-to-churn customer is usually costlier than a false alarm) — try `MIN_RECALL = 0.65` and `MIN_ROC_AUC = 0.80` as a starting point, and adjust after seeing the actual candidate scores below.

In [ ]:
MIN_RECALL = ...    # TODO: e.g. 0.65
MIN_ROC_AUC = ...   # TODO: e.g. 0.80


def passes_quality_gate(recall: float, roc_auc: float) -> bool:
    # TODO: return True only if both thresholds are met
    return ...

---
## Part 3: Log Every Candidate as an MLflow Run

### TODO 2 — Wrap each candidate model's fit + evaluation in `with mlflow.start_run(...)`

**HINT:** for each candidate, inside the `with` block:
1. `mlflow.log_param("model_type", name)` and `mlflow.log_params(model.get_params())`
2. Fit the full pipeline on `X_train`/`y_train`, predict on `X_test`
3. Compute `recall_score`, `precision_score`, `roc_auc_score` (needs `predict_proba`), `f1_score` and log each with `mlflow.log_metric`
4. Compute `gate_passed = passes_quality_gate(recall, roc_auc)` and log it with `mlflow.set_tag("passes_quality_gate", gate_passed)`
5. `mlflow.sklearn.log_model(pipeline, artifact_path="model")` — logs the **entire fitted `Pipeline`**, not just the classifier

In [ ]:
candidate_models = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=RNG_SEED),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=RNG_SEED),
    "gradient_boosting": GradientBoostingClassifier(random_state=RNG_SEED),
}

if MLFLOW_AVAILABLE:
    for name, model in candidate_models.items():
        with mlflow.start_run(run_name=name):
            pipeline = Pipeline([
                ("risk_flag", RecencyRiskFlagger()),
                ("preprocessor", preprocessor),
                ("model", model),
            ])

            # TODO: log params, fit, predict, log metrics + quality-gate tag, log the model
            mlflow.log_param("model_type", name)
            mlflow.log_params(model.get_params())

            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            y_proba = pipeline.predict_proba(X_test)[:, 1]

            recall = ...      # recall_score(y_test, y_pred)
            precision = ...   # precision_score(y_test, y_pred)
            roc_auc = ...      # roc_auc_score(y_test, y_proba)
            f1 = ...           # f1_score(y_test, y_pred)
            gate_passed = ...  # passes_quality_gate(recall, roc_auc)

            mlflow.log_metric("recall", recall)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("roc_auc", roc_auc)
            mlflow.log_metric("f1", f1)
            mlflow.set_tag("passes_quality_gate", gate_passed)

            mlflow.sklearn.log_model(pipeline, artifact_path="model")

            print(f"{name}: recall={recall:.3f} precision={precision:.3f} roc_auc={roc_auc:.3f} "
                  f"f1={f1:.3f} gate_passed={gate_passed}")
else:
    print("Skipped — install mlflow to run this cell.")

---
## Part 4: Compare Runs and Select the Best Gate-Passing Candidate

Launch `mlflow ui` from a terminal in this project's root (`mlflow ui` → `http://localhost:5000`) to browse runs interactively — the parallel coordinates plot is particularly useful here for seeing how `model_type` relates to `roc_auc` across all three candidates at a glance. Programmatically:

### TODO 3 — Query runs, filter to gate-passing ones, and pick the best by ROC-AUC

**HINT:** `mlflow.search_runs(experiment_names=[EXPERIMENT_NAME], filter_string="tags.passes_quality_gate = 'True'", order_by=["metrics.roc_auc DESC"])`.

In [ ]:
if MLFLOW_AVAILABLE:
    # TODO: search_runs filtered to gate-passing runs, ordered by roc_auc descending
    runs_df = ...
    print(runs_df[["tags.mlflow.runName", "metrics.recall", "metrics.precision", "metrics.roc_auc", "metrics.f1"]].to_string(index=False))

    best_run_id = runs_df.iloc[0]["run_id"]
    best_run_name = runs_df.iloc[0]["tags.mlflow.runName"]
    print(f"\nBest gate-passing run: {best_run_name} ({best_run_id})")
else:
    print("Skipped — install mlflow to run this cell.")

---
## Part 5: Register the Winning Pipeline and Promote It

Registering creates a new, independently-versioned entry in the Model Registry, pointing back at the exact run that produced it. Promotion through the stage lifecycle (`None → Staging → Production → Archived`) is what a serving layer actually reads from — never a raw run ID.

### TODO 4 — Register and promote to `Staging`, then `Production`

**HINT:** `mlflow.register_model(model_uri=f"runs:/{best_run_id}/model", name="customer-churn-classifier")` returns an object with a `.version` attribute; then use `MlflowClient().transition_model_version_stage(name=..., version=..., stage="Staging")`, inspect it, and transition again to `"Production"`.

In [ ]:
if MLFLOW_AVAILABLE:
    REGISTERED_MODEL_NAME = "customer-churn-classifier"

    # TODO: register the best run's model
    registration_result = ...  # mlflow.register_model(model_uri=f"runs:/{best_run_id}/model", name=REGISTERED_MODEL_NAME)

    client = MlflowClient()
    # TODO: transition to "Staging", then to "Production"
    client.transition_model_version_stage(
        name=REGISTERED_MODEL_NAME, version=registration_result.version, stage="Staging",
    )
    client.transition_model_version_stage(
        name=REGISTERED_MODEL_NAME, version=registration_result.version, stage="Production",
    )
    print(f"{REGISTERED_MODEL_NAME} v{registration_result.version} is now in Production")
else:
    print("Skipped — install mlflow to run this cell.")

---
## Part 6: What the Serving Layer Actually Loads

This project's `src/api/` (FastAPI, real-time single-customer scoring) and `src/cli/` (scheduled batch prediction) both load the model the same way — **by registry stage, never by run ID or file path**:

```python
production_model = mlflow.sklearn.load_model("models:/customer-churn-classifier/Production")
predictions = production_model.predict(new_customers_df)  # raw, unprocessed input — the Pipeline handles the rest
```

Promoting a retrained model to `Production` is then a **registry operation** — no redeploy, no code change in `src/api/` or `src/cli/` required. Rolling back is transitioning a previous version back to `Production`.

In [ ]:
if MLFLOW_AVAILABLE:
    production_model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}/Production")
    sample_predictions = production_model.predict_proba(X_test.head(5))[:, 1]
    print("Sample churn probabilities from the registry's Production model:")
    print(np.round(sample_predictions, 3))
else:
    print("Skipped — install mlflow to run this cell.")

---
## Senior Engineer Notes & Best Practices

1. **Log every candidate, gate-passing or not.** The two models that *didn't* clear the bar are valuable historical evidence — next quarter, when someone asks "did we already try gradient boosting for this?", the answer is in MLflow, not in someone's memory.
2. **Define the quality gate as code, once, and apply it identically to every run** — `passes_quality_gate()` being a single function called inside the loop is what prevents "we sort of eyeballed it and it looked fine" from ever becoming the actual promotion criteria.
3. **`mlflow.sklearn.log_model` on the *whole pipeline*, never just the bare classifier.** If you log only `model` instead of `pipeline`, the registry's artefact silently stops matching this project's fundamental design principle: one deployable object that accepts raw input.
4. **Registry stage, not run ID, is what serving code should reference.** This is what makes "deploy a retrained model" a metadata change instead of a code change — and what makes rollback a one-line operation instead of a redeploy.
5. **Repeated logic across notebooks (the custom transformer, the `ColumnTransformer`) is a strong signal it belongs in `src/pipeline/`**, imported by notebooks, the FastAPI service, and the CLI alike — notebooks are for exploration and learning, `src/` is what actually ships. That extraction is this project's natural next step once you're comfortable with everything built across these five notebooks.

## Key Takeaways
- MLflow Tracking answers "what did we try, and what happened" for every candidate; the Model Registry answers "what's actually live right now"
- A quality gate must be one deterministic function applied identically to every run, not a per-run judgment call
- Promotion to `Production` is a registry-level decision, decoupled entirely from deploying new code in `src/api/` or `src/cli/`
- This notebook closes the loop the README promises: segmentation (Notebook 1) → visualization (Notebook 2) → leak-free churn pipeline (Notebook 3) → tracked, registered, promotable production candidate (this notebook)